# Borjas, *Labor Economics* (9th ed.), Chapter 2: Labor Supply

Models only (sections 2-3 to 2-7, 2-9, 2-11 to 2-14); empirical sections, "Theory at Work" boxes and Data Explorers are skipped.

## Summary

| Section | Model | Key result |
|---|---|---|
| 2-3, 2-4 | Preferences over consumption $C$ and leisure $L$; budget line $C = w(T-L)+V$ | Indifference curves are downward sloping and convex; the budget line has slope $-w$ and passes through the endowment point $(T, V)$ |
| 2-5 | Utility maximization | Interior solution at the tangency $MU_L/MU_C = w$ |
| 2-5 | Comparative statics | Income effect lowers hours (leisure is normal); substitution effect raises them; the wage effect is ambiguous |
| 2-6 | Participation | Work iff the market wage exceeds the **reservation wage** $w_R = MU_L/MU_C$ evaluated at $h=0$ |
| 2-7 | Labor supply curve | $h^*(w)$ slopes up when substitution dominates, bends backward when income dominates |
| 2-9 | Household production | Time has a shadow price; higher $w$ shifts time from home production to the market |
| 2-11, 2-12, 2-14 | Welfare, EITC, disability | Policies create kinks and non-convexities in the budget set |
| 2-13 | Life cycle | Temporary wage changes move hours more than permanent ones (intertemporal substitution) |

## Assumptions and notation

| Symbol | Meaning | Baseline |
|---|---|---|
| $C$ | Consumption, dollar value of all goods bought per week | |
| $L$ | Leisure hours per week | |
| $h$ | Hours worked, $h = T - L$ | |
| $T$ | Total non-sleeping hours per week | 110 |
| $w$ | Hourly wage (constant, taken as given) | 10 |
| $V$ | Non-labor income per week | 100 |
| $U = f(C, L)$ | Utility function | $L^{a} C^{1-a}$ |
| $a$ | Leisure weight | 0.6 |
| $MU_L, MU_C$ | Marginal utilities | |
| $MRS = MU_L/MU_C$ | Marginal rate of substitution (absolute slope of an indifference curve) | |
| $w_R$ | Reservation wage | |
| $\sigma$ | Elasticity of substitution between $L$ and $C$ (CES variant) | 0.5 |

The baseline follows the example in section 2-5 ($V = 100$, $w = 10$, $T = 110$).

**Assumptions**

1. A single representative person, one period, all goods aggregated into $C$ with price 1.
2. $C$ and $L$ are goods: $MU_C, MU_L > 0$.
3. Indifference curves are convex (diminishing MRS).
4. The wage is constant: no overtime premium, no hours-dependent wage.
5. Leisure is a normal good.
6. The person maximizes utility subject to $C = wh + V$ and $h + L = T$.
7. **Functional form for the code:** Cobb-Douglas, $U = L^{a}C^{1-a}$. This satisfies 2, 3 and 5 but has a special feature that the book does not assume: with $V = 0$ the income and substitution effects of a wage change cancel exactly. A CES variant is used below to show backward-bending supply.

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from scipy.optimize import minimize, minimize_scalar, brentq

from common import *

P = supply
P

: 

## 1. Preferences, budget constraint and the hours decision (2-3 to 2-5)

With $U = L^{a}C^{1-a}$:

$$MU_L = a\,\frac{U}{L}, \qquad MU_C = (1-a)\,\frac{U}{C}, \qquad MRS = \frac{MU_L}{MU_C} = \frac{a}{1-a}\,\frac{C}{L}.$$

The worker solves $\max_{L,C} L^a C^{1-a}$ subject to $C = w(T-L) + V$. The tangency condition (2-9) $MRS = w$ together with the budget line gives the solution; sympy derives it below.

In [ ]:
L, Cc, w, V, T, a = sp.symbols("L C w V T a", positive=True)
U = L**a * Cc ** (1 - a)
MU_L, MU_C = sp.diff(U, L), sp.diff(U, Cc)

tangency = sp.Eq(sp.simplify(MU_L / MU_C), w)
budget = sp.Eq(Cc, w * (T - L) + V)
sol = sp.solve([tangency, budget], [L, Cc], dict=True)[0]
L_star, C_star = sp.simplify(sol[L]), sp.simplify(sol[Cc])
h_star = sp.simplify(T - L_star)

display(sp.Eq(sp.Symbol("L^*"), L_star))
display(sp.Eq(sp.Symbol("C^*"), C_star))
display(sp.Eq(sp.Symbol("h^*"), sp.expand(h_star)))

So $L^* = a\,(wT+V)/w$: the worker spends a fraction $a$ of *full income* $wT+V$ on leisure, and $h^* = (1-a)T - aV/w$.

Now solve the same problem numerically with `scipy` (choose $h \in [0,T]$ to maximize utility) and compare with the closed form.

In [ ]:
def h_closed(w, V=P.V, a=P.a, T=P.T):
    # Cobb-Douglas hours, floored at zero (corner solution)
    return np.maximum(0.0, (1 - a) * T - a * V / np.asarray(w, dtype=float))


def h_numeric(w, V=P.V, a=P.a, T=P.T):
    res = minimize_scalar(
        lambda h: -cobb_douglas(T - h, w * h + V, a),
        bounds=(0, T - 1e-9), method="bounded", options={"xatol": 1e-10},
    )
    return res.x


for wage in [5, 10, 20, 40]:
    print(f"w={wage:>3}: closed form {h_closed(wage):7.3f}  numeric {h_numeric(wage):7.3f}")

assert np.allclose([h_closed(x) for x in [5, 10, 20, 40]], [h_numeric(x) for x in [5, 10, 20, 40]], atol=1e-4)

h0 = float(h_closed(P.w)); L0 = P.T - h0; C0 = P.w * h0 + P.V
print(f"\nBaseline optimum: h* = {h0:.1f} hours, L* = {L0:.1f}, C* = ${C0:.0f}")

In [ ]:
def plot_budget_and_ic(ax, w, V, a, T, levels=(), label=None, color=BLUE, mark="P"):
    Lg = np.linspace(1, T, 400)
    ax.plot(Lg, w * (T - Lg) + V, color=color, lw=2, label=label)
    Ls = a * (w * T + V) / w; Cs = (1 - a) * (w * T + V)
    u_star = cobb_douglas(Ls, Cs, a)
    for u in list(levels) + [u_star]:
        ax.plot(Lg, indifference_curve(u, a, Lg), color=GREY, lw=1, ls="--" if u != u_star else "-")
    ax.plot([Ls], [Cs], "o", color=RED, zorder=5)
    ax.annotate(mark, (Ls, Cs), textcoords="offset points", xytext=(6, 6))
    ax.plot([T], [V], "s", color="k", ms=5); ax.annotate("E", (T, V), textcoords="offset points", xytext=(-12, 6))
    ax.set_xlim(0, T + 3); ax.set_ylim(0, w * T + V + 100)
    ax.set_xlabel("Leisure L (hours/week)"); ax.set_ylabel("Consumption C ($/week)")
    return u_star


fig, ax = plt.subplots()
plot_budget_and_ic(ax, P.w, P.V, P.a, P.T, levels=[0.7 * cobb_douglas(L0, C0, P.a), 1.3 * cobb_douglas(L0, C0, P.a)])
ax.set_title("Interior solution: tangency of indifference curve and budget line (Figure 2-6)")
plt.show()

## 2. Comparative statics: income and substitution effects (2-5)

**Nonlabor income.** $\partial h^*/\partial V = -a/w < 0$: a pure income effect, leisure is normal.

**Wage.** $\partial h^*/\partial w = aV/w^2 \ge 0$. The Slutsky equation splits the wage effect into

$$\underbrace{\frac{\partial L}{\partial w}}_{\text{total}} = \underbrace{\frac{\partial L^c}{\partial w}}_{\text{substitution } (<0)} + \underbrace{(T - L)\,\frac{\partial L}{\partial V}}_{\text{income}} .$$

Here $T-L = h$ is the worker's initial hours (her *labor* endowment sold at wage $w$), which is why a wage increase is an income gain for someone who works. With Cobb-Douglas and $V=0$ the two pieces cancel exactly, so hours do not respond to the wage at all.

The compensated (Hicksian) leisure demand holds utility at $U_0$:
$L^c(w, U_0) = U_0\,\big(a / ((1-a)w)\big)^{1-a}$. The code below reproduces Figure 2-9 for a wage increase from \$10 to \$20: P is the original optimum, R the new one, and Q the compensated point (new slope, old indifference curve).

In [ ]:
w0, w1 = P.w, 2 * P.w
U0 = cobb_douglas(L0, C0, P.a)

# Compensated leisure at the new wage, holding utility fixed at U0
L_Q = U0 * (P.a / ((1 - P.a) * w1)) ** (1 - P.a)
C_Q = indifference_curve(U0, P.a, L_Q)
h_Q = P.T - L_Q

h_R = float(h_closed(w1)); L_R = P.T - h_R; C_R = w1 * h_R + P.V

print(f"P: h = {h0:5.1f}  (L = {L0:5.1f})")
print(f"Q: h = {h_Q:5.1f}  (L = {L_Q:5.1f})   substitution effect on hours = {h_Q - h0:+.1f}")
print(f"R: h = {h_R:5.1f}  (L = {L_R:5.1f})   income effect on hours       = {h_R - h_Q:+.1f}")
print(f"Total effect of the wage increase = {h_R - h0:+.1f} hours")

# Check Q against a numerical constrained minimization of expenditure
res = minimize(lambda z: z[0] * w1 + z[1], x0=[60, 400],
               constraints={"type": "eq", "fun": lambda z: cobb_douglas(z[0], z[1], P.a) - U0},
               bounds=[(1e-6, P.T), (1e-6, None)])
assert np.allclose(res.x, [L_Q, C_Q], rtol=1e-3)

fig, ax = plt.subplots(figsize=(7, 4.6))
Lg = np.linspace(1, P.T, 400)
ax.plot(Lg, w0 * (P.T - Lg) + P.V, color=BLUE, lw=2, label="old budget line (w=10)")
ax.plot(Lg, w1 * (P.T - Lg) + P.V, color=GREEN, lw=2, label="new budget line (w=20)")
ax.plot(Lg, C_Q - w1 * (Lg - L_Q), color=ORANGE, lw=1.5, ls=":", label="compensated line (slope -20)")
ax.plot(Lg, indifference_curve(U0, P.a, Lg), color=GREY, label="$U_0$")
ax.plot(Lg, indifference_curve(cobb_douglas(L_R, C_R, P.a), P.a, Lg), color=GREY, ls="--", label="$U_1$")
for name, (x, y) in {"P": (L0, C0), "Q": (L_Q, C_Q), "R": (L_R, C_R)}.items():
    ax.plot([x], [y], "o", color=RED); ax.annotate(name, (x, y), textcoords="offset points", xytext=(6, 6))
ax.set_xlim(0, P.T + 2); ax.set_ylim(0, 1500)
ax.set_xlabel("Leisure"); ax.set_ylabel("Consumption"); ax.legend(fontsize=8)
ax.set_title("Income and substitution effects of a wage increase (Figure 2-9)")
plt.show()

## 3. To work or not to work? (2-6)

The person works only if the wage exceeds the **reservation wage**, the MRS at the corner $h=0$, $(L,C)=(T,V)$:

$$w_R = \frac{MU_L}{MU_C}\Big|_{(T,V)} = \frac{a}{1-a}\,\frac{V}{T}.$$

If working also requires a fixed cost $F$ (commuting, child care), the budget line has a jump at $h=0$ and the worker compares the **utility** at the two options. The wage at which the two are equal is the reservation wage with fixed costs; hours jump from 0 to a positive number there.

In [ ]:
F = 40.0  # weekly fixed cost of working

w_R = P.a / (1 - P.a) * P.V / P.T
print(f"Reservation wage without fixed cost: {w_R:.2f}")

def utility_work(w, F=0.0):
    # work optimally given a fixed cost F: nonlabor income net of F
    h = max(0.0, (1 - P.a) * P.T - P.a * (P.V - F) / w)
    return cobb_douglas(P.T - h, w * h + P.V - F, P.a), h

U_no_work = cobb_douglas(P.T, P.V, P.a)
w_RF = brentq(lambda x: utility_work(x, F)[0] - U_no_work, 0.5, 50)
print(f"Reservation wage with F = {F:.0f}:      {w_RF:.2f}")

ws = np.linspace(0.5, 30, 600)
h_nofix = h_closed(ws)
h_fix = np.array([utility_work(x, F)[1] if x >= w_RF else 0.0 for x in ws])

fig, ax = plt.subplots()
ax.plot(h_nofix, ws, color=BLUE, label="no fixed cost")
ax.plot(h_fix, ws, color=RED, label=f"fixed cost F = {F:.0f}")
ax.set_xlabel("Hours worked"); ax.set_ylabel("Wage"); ax.legend()
ax.set_title("Labor supply with and without a fixed cost: hours jump at $w_R$")
plt.show()

## 4. The labor supply curve and its elasticity (2-7)

The supply elasticity is $\varepsilon = \dfrac{d\ln h}{d\ln w}$. For Cobb-Douglas it is $\dfrac{aV}{w\,h^*}$, which is small and positive for these parameters.

A backward-bending supply curve needs an income effect that eventually dominates. Cobb-Douglas cannot deliver this when $V > 0$ (hours rise towards the asymptote $(1-a)T$), so we use a CES utility function

$$U = \big[a L^{\rho} + (1-a) C^{\rho}\big]^{1/\rho}, \quad \sigma = \frac{1}{1-\rho},$$

for which leisure demand at prices $(w, 1)$ and full income $I = wT+V$ is

$$L^* = \frac{I\,(a/w)^{\sigma}}{a^{\sigma} w^{1-\sigma} + (1-a)^{\sigma}} .$$

With $\sigma<1$ (income effect strong relative to substitution) hours eventually fall as $w$ rises.

In [ ]:
def h_ces(w, sigma, V=P.V, a=P.a, T=P.T):
    w = np.asarray(w, dtype=float); I = w * T + V
    Lstar = I * (a / w) ** sigma / (a**sigma * w ** (1 - sigma) + (1 - a) ** sigma)
    return np.maximum(0.0, T - Lstar)


def h_ces_numeric(w, sigma, V=P.V, a=P.a, T=P.T):
    rho = 1 - 1 / sigma
    u = lambda h: -((a * (T - h) ** rho + (1 - a) * (w * h + V) ** rho) ** (1 / rho))
    return minimize_scalar(u, bounds=(0, T - 1e-9), method="bounded", options={"xatol": 1e-10}).x


for wage in [5, 20, 80]:
    for s in [0.5, 2.0]:
        assert abs(h_ces(wage, s) - h_ces_numeric(wage, s)) < 1e-3

ws = np.linspace(2, 150, 500)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(h_closed(ws), ws, color=BLUE, label="Cobb-Douglas")
for s, c in [(0.5, RED), (2.0, GREEN)]:
    axes[0].plot(h_ces(ws, s), ws, color=c, label=fr"CES $\sigma$={s}")
axes[0].set_xlabel("Hours"); axes[0].set_ylabel("Wage"); axes[0].legend(); axes[0].set_title("Labor supply curves")

# Elasticity of the Cobb-Douglas curve: finite difference vs closed form
eps_fd = (np.log(h_closed(ws * 1.001)) - np.log(h_closed(ws))) / np.log(1.001)
eps_cf = P.a * P.V / (ws * h_closed(ws))
assert np.allclose(eps_fd[ws > 5], eps_cf[ws > 5], atol=1e-3)
axes[1].plot(ws, eps_cf, color=BLUE); axes[1].set_xlabel("Wage"); axes[1].set_ylabel("Elasticity of hours")
axes[1].set_title("Cobb-Douglas supply elasticity"); axes[1].set_ylim(0, 0.5)
plt.tight_layout(); plt.show()
print(f"Elasticity at w=10: {P.a * P.V / (P.w * h0):.3f}")

## 5. Household production (2-9)

The person combines market goods $x$ and home time $H$ to produce a home-consumed good $Z = x^{\beta}H^{1-\beta}$ and has utility $U = Z^{\gamma}L^{1-\gamma}$. Constraints: $x = wh + V$ and $h + H + L = T$, so all time and income collapse into one *full-income* budget

$$x + wH + wL = wT + V = I .$$

Home time is priced at the wage. The cost-minimizing way to produce one unit of $Z$ costs

$$\pi(w) = \Big(\tfrac{1}{\beta}\Big)^{\beta}\Big(\tfrac{w}{1-\beta}\Big)^{1-\beta},$$

and with Cobb-Douglas utility $Z = \gamma I/\pi$, $L = (1-\gamma)I/w$, $H = (1-\beta)\pi Z/w$, $x = \beta\pi Z$, $h = T - L - H$. A higher wage makes home time expensive: $H$ falls, so market hours rise, **even with $V=0$**.

In [ ]:
def household(w, V=P.V, T=P.T, beta=P.beta, gamma=P.gamma):
    w = np.asarray(w, dtype=float); I = w * T + V
    pi = (1 / beta) ** beta * (w / (1 - beta)) ** (1 - beta)
    Z = gamma * I / pi
    Lh = (1 - gamma) * I / w
    H = (1 - beta) * pi * Z / w
    x = beta * pi * Z
    return dict(L=Lh, H=H, x=x, h=T - Lh - H, Z=Z)


# Numerical check with SLSQP over (x, H, L)
def household_numeric(w):
    f = lambda z: -((z[0] ** P.beta * z[1] ** (1 - P.beta)) ** P.gamma * z[2] ** (1 - P.gamma))
    con = {"type": "eq", "fun": lambda z: z[0] + w * z[1] + w * z[2] - (w * P.T + P.V)}
    r = minimize(f, [300, 30, 40], constraints=con, bounds=[(1e-6, None)] * 3, method="SLSQP")
    return r.x

hh = household(P.w)
x_n, H_n, L_n = household_numeric(P.w)
assert np.allclose([hh["x"], hh["H"], hh["L"]], [x_n, H_n, L_n], rtol=1e-3)
print({k: round(float(v), 1) for k, v in hh.items()})

ws = np.linspace(2, 60, 200); res = household(ws)
fig, ax = plt.subplots()
ax.plot(ws, res["h"], color=BLUE, label="market hours h"); ax.plot(ws, res["H"], color=GREEN, label="home production H")
ax.plot(ws, res["L"], color=ORANGE, label="leisure L")
ax.set_xlabel("Wage"); ax.set_ylabel("Hours per week"); ax.legend()
ax.set_title("Time allocation with household production")
plt.show()

## 6. Life-cycle labor supply (2-13)

Two periods $t=1,2$, utility $\sum_t \beta^{t-1}\big[a\ln L_t + (1-a)\ln C_t\big]$ (discount factor $\beta$, here `disc`), wages $w_t$, interest rate $r$, initial assets $A_0$, lifetime budget

$$C_1 + \frac{C_2}{1+r} + w_1L_1 + \frac{w_2L_2}{1+r} = W \equiv A_0 + w_1T + \frac{w_2T}{1+r}.$$

Let $\delta = \beta(1+r)$. First-order conditions give

$$L_1 = \frac{a\,W}{(1+\beta)\,w_1}, \qquad L_2 = \frac{a\,\delta\,W}{(1+\beta)\,w_2}.$$

A **temporary** wage rise in period 1 raises $w_1$ but barely changes lifetime wealth $W$ (small income effect), so hours in period 1 rise a lot and hours in period 2 fall. A **permanent** rise raises $w_1$ and $w_2$ proportionally; with log utility and $A_0=0$, $L_t$ depends only on $W/w_t$, so the income effect exactly cancels the substitution effect and hours do not change.

In [ ]:
def lifecycle(w1, w2, T=P.T, a=P.a, beta=P.disc, r=P.r, A0=P.A0):
    W = A0 + w1 * T + w2 * T / (1 + r)
    delta = beta * (1 + r)
    L1 = a * W / ((1 + beta) * w1)
    L2 = a * delta * W / ((1 + beta) * w2)
    return T - L1, T - L2


def lifecycle_numeric(w1, w2, T=P.T, a=P.a, beta=P.disc, r=P.r, A0=P.A0):
    # variables are scaled by 100 so that SLSQP is well conditioned
    W = (A0 + w1 * T + w2 * T / (1 + r)) / 100
    f = lambda z: -(a * np.log(z[0]) + (1 - a) * np.log(z[2]) + beta * (a * np.log(z[1]) + (1 - a) * np.log(z[3])))
    con = {"type": "eq", "fun": lambda z: z[2] + z[3] / (1 + r) + w1 * z[0] + w2 * z[1] / (1 + r) - W}
    z = minimize(f, [0.6, 0.6, 4, 4], constraints=con, bounds=[(1e-3, T / 100)] * 2 + [(1e-3, None)] * 2,
                 method="SLSQP", options={"ftol": 1e-14, "maxiter": 500}).x * 100
    return T - z[0], T - z[1]


base = lifecycle(10, 10)
temp = lifecycle(12, 10)
perm = lifecycle(12, 12)
for name, args, res_ in [("baseline", (10, 10), base), ("temporary +20% (period 1)", (12, 10), temp), ("permanent +20%", (12, 12), perm)]:
    assert np.allclose(res_, lifecycle_numeric(*args), atol=0.05)
    print(f"{name:<28} h1 = {res_[0]:6.2f}  h2 = {res_[1]:6.2f}")

print(f"\nChange in period-1 hours: temporary {temp[0]-base[0]:+.2f}, permanent {perm[0]-base[0]:+.2f}")

## 7. Policy: kinked budget constraints (2-11, 2-12, 2-14)

These sections change the budget set, not preferences. The budget is no longer a line, so we maximize utility over a grid of hours instead of using the tangency condition (the budget set may be non-convex).

* **Welfare (2-11):** a guarantee $G$ is reduced by a fraction $t$ of each earned dollar: $C = wh + V + \max(0,\, G - t\,wh)$. With $t=1$ every earned dollar below $G$ is taxed away (flat segment of the budget line).
* **EITC (2-12):** a credit of $s \times$ earnings up to $Y_1$ (phase-in), constant up to $Y_2$ (plateau), then reduced at rate $\pi_p$ until it is exhausted (phase-out).

In [ ]:
h_grid = np.linspace(0, P.T - 0.01, 4000)
earn = P.w * h_grid


def consumption_welfare(G, t):
    return earn + P.V + np.maximum(0.0, G - t * earn)


def consumption_eitc(s=0.4, Y1=150.0, Y2=250.0, pp=0.2):
    credit_max = s * Y1
    credit = np.where(earn <= Y1, s * earn, np.where(earn <= Y2, credit_max, np.maximum(0.0, credit_max - pp * (earn - Y2))))
    return earn + P.V + credit


def best(Cgrid):
    u = cobb_douglas(P.T - h_grid, np.maximum(Cgrid, 1e-9), P.a)
    i = np.argmax(u)
    return h_grid[i], Cgrid[i]


scenarios = {
    "no program": earn + P.V,
    "welfare G=250, t=100%": consumption_welfare(250, 1.0),
    "welfare G=250, t=50%": consumption_welfare(250, 0.5),
    "EITC": consumption_eitc(),
}
fig, ax = plt.subplots(figsize=(7, 4.6))
for (name, Cg), col in zip(scenarios.items(), [GREY, RED, ORANGE, GREEN]):
    h_opt, C_opt = best(Cg)
    ax.plot(P.T - h_grid, Cg, color=col, label=f"{name} (h* = {h_opt:.1f})")
    ax.plot([P.T - h_opt], [C_opt], "o", color=col)
    print(f"{name:<26} h* = {h_opt:5.1f}  C* = {C_opt:6.1f}")
ax.set_xlabel("Leisure"); ax.set_ylabel("Consumption"); ax.legend(fontsize=8)
ax.set_title("Budget sets with welfare and the EITC")
plt.show()

## Check list

* Tangency $MRS = w$ holds at the optimum: the closed form, sympy and scipy agree (asserts above).
* Life cycle: a temporary wage rise raises current hours by several hours; a permanent one leaves them unchanged under log utility.
* Cobb-Douglas with $V>0$ gives a mildly upward-sloping supply curve, $\varepsilon = aV/(wh^*) \approx 0.16$ at the baseline.
* Income and substitution effects: the P to Q to R decomposition reproduces Figure 2-9.
* Backward-bending supply needs $\sigma<1$ (CES).